# MNIST layer ablation

Every baseline and ablated layer is cached separately with `UniversalDumper`, so restarting resumes at the next missing training run.

In [ ]:
import copy
import typing

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torchvision
import torchvision.transforms.v2 as v2

import zigzag.utils
from zigzag.pipelines.validate import train_validate, validate_pretrained

LEARNING_RATE = 1e-5


def make_vit_b_32(head: torch.nn.Module) -> torch.nn.Module:
    model = torchvision.models.vit_b_32(num_classes=1000, weights=torchvision.models.ViT_B_32_Weights.DEFAULT)
    model.heads = head
    return model


def make_vit_l_16(head: torch.nn.Module) -> torch.nn.Module:
    model = torchvision.models.vit_l_16(num_classes=1000, weights=torchvision.models.ViT_L_16_Weights.DEFAULT)
    model.heads = head
    return model


def make_resnet34(head: torch.nn.Module) -> torch.nn.Module:
    model = torchvision.models.resnet34(num_classes=1000, weights=torchvision.models.ResNet34_Weights.DEFAULT)
    model.fc = head
    return model


MODEL_CONFIGS = {
    'vit_b_32': (make_vit_b_32, torchvision.models.ViT_B_32_Weights.DEFAULT.transforms),
    'vit_l_16': (make_vit_l_16, torchvision.models.ViT_L_16_Weights.DEFAULT.transforms),
    'resnet34': (make_resnet34, torchvision.models.ResNet34_Weights.DEFAULT.transforms),
}


In [ ]:
class ResidualBlockBypass(torch.nn.Module):
    """Remove a residual block while retaining its required shape-changing shortcut."""
    def __init__(self, block: torch.nn.Module):
        super().__init__()
        self.downsample = block.downsample

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x if self.downsample is None else self.downsample(x)


def hidden_state_layers(model: torch.nn.Module) -> list[tuple[str, tuple[int, ...]]]:
    """Removable blocks, in exactly the order yielded after the initial state."""
    if isinstance(model, torchvision.models.VisionTransformer):
        # yield_hidden_states yields the embedding first, then each encoder block.
        return [(f'encoder.layers.{i}', (i,)) for i, _ in enumerate(model.encoder.layers)]
    if isinstance(model, torchvision.models.ResNet):
        # The initial max-pool state has no module to remove; remaining states are blocks below.
        return [(f'layer{stage}.{block}', (stage, block))
                for stage in range(1, 5)
                for block, _ in enumerate(getattr(model, f'layer{stage}'))]
    raise TypeError(f'Unsupported architecture: {type(model)}')


def remove_hidden_state_layer(model: torch.nn.Module, location: tuple[int, ...]) -> None:
    if isinstance(model, torchvision.models.VisionTransformer):
        (block,) = location
        model.encoder.layers[block] = torch.nn.Identity()
    elif isinstance(model, torchvision.models.ResNet):
        stage, block = location
        blocks = getattr(model, f'layer{stage}')
        blocks[block] = ResidualBlockBypass(blocks[block])
    else:
        raise TypeError(f'Unsupported architecture: {type(model)}')


def mnist_datasets(transforms_factory):
    transforms = v2.Compose([v2.Grayscale(num_output_channels=3), transforms_factory()])
    return (torchvision.datasets.MNIST('mnist', train=True, download=True, transform=transforms),
            torchvision.datasets.MNIST('mnist', train=False, download=True, transform=transforms))


def final_accuracy(history: pd.DataFrame) -> float:
    for column in ('Accuracy', 'accuracy'):
        if column in history:
            return float(history[column].iloc[-1])
    raise KeyError(f'No accuracy column in training history: {list(history.columns)}')


def load_or_train(dumper, model, train_ds, test_ds):
    # A history without its model means an interrupted run; do not mistake the fresh model for a checkpoint.
    if dumper.has_dump('trained_model'):
        return dumper.get_dump('trained_model')
    if dumper.has_dump('train_model_history'):
        raise RuntimeError(f'{dumper.directory_} has a history but no checkpoint; remove this incomplete directory and rerun.')
    train_validate(model, train_ds, test_ds, dumper, learning_rate=LEARNING_RATE)
    return dumper.get_dump('trained_model')


In [ ]:
def load_or_train_baseline(model_name, make_model, train_ds, test_ds):
    model_dumper = zigzag.utils.UniversalDumper(f'ablation_results/mnist/{model_name}')
    pretrained_dumper = model_dumper.make_subdumper('pretrained')
    finetuned_dumper = model_dumper.make_subdumper('finetuned')
    if not pretrained_dumper.has_dump('trained_head'):
        validate_pretrained(make_model(torch.nn.Identity()), train_ds, train_ds.targets,
                            test_ds, test_ds.targets, pretrained_dumper)
    baseline = load_or_train(finetuned_dumper, make_model(pretrained_dumper.get_dump('trained_head')),
                             train_ds, test_ds)
    return model_dumper, baseline, final_accuracy(finetuned_dumper.get_dump('train_model_history'))


def run_ablations(model_name, make_model, transforms_factory) -> pd.DataFrame:
    train_ds, test_ds = mnist_datasets(transforms_factory)
    model_dumper, baseline, baseline_accuracy = load_or_train_baseline(
        model_name, make_model, train_ds, test_ds
    )
    rows = []
    for index, (layer_name, location) in enumerate(hidden_state_layers(baseline), start=1):
        # Per-layer directories are the restart checkpoints, including model and full history.
        layer_dumper = model_dumper.make_subdumper(f'ablations/{index:02d}_{layer_name}')
        ablated_model = copy.deepcopy(baseline)
        remove_hidden_state_layer(ablated_model, location)
        load_or_train(layer_dumper, ablated_model, train_ds, test_ds)
        accuracy = final_accuracy(layer_dumper.get_dump('train_model_history'))
        rows.append({'model': model_name, 'layer_index': index, 'layer': layer_name,
                     'baseline_accuracy': baseline_accuracy, 'ablated_accuracy': accuracy,
                     'accuracy_degradation': baseline_accuracy - accuracy})
    results = pd.DataFrame(rows)
    model_dumper.save_dump(results, 'ablation_summary')
    return results


results = pd.concat([run_ablations(name, *config) for name, config in MODEL_CONFIGS.items()],
                    ignore_index=True)
results


In [ ]:
fig, axes = plt.subplots(len(MODEL_CONFIGS), 1, figsize=(15, 11), constrained_layout=True)
for axis, (model_name, model_results) in zip(axes, results.groupby('model', sort=False)):
    axis.bar(model_results['layer'], model_results['accuracy_degradation'], color='tab:red')
    axis.axhline(0, color='black', linewidth=0.8)
    axis.set_title(f'{model_name}: final MNIST accuracy degradation after one-layer ablation')
    axis.set_ylabel('baseline accuracy − ablated accuracy')
    axis.tick_params(axis='x', rotation=45)
axes[-1].set_xlabel('removed block (same order as yield_hidden_states)')
plt.show()
